# Advanced Lab · VLA / World-Action Interface

Optional lab. VLA/WA/π0 类工作把视觉状态映射到动作 chunk，或学习 action/world consequences。这里把它限制在已有 planner/safety interface：action proposal 必须带时间窗、frame、uncertainty 和可拒绝条件。

In [ ]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "src").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
sys.path.insert(0, str(PROJECT_ROOT / "src"))

from ad_tutorial import (
    ARTIFACT_DIR,
    BEVConfig,
    build_bev_dataset,
    build_urban_cut_in_scene,
    ensure_artifact_dir,
    load_json_artifact,
    load_numpy_artifact,
    save_json_artifact,
    save_numpy_artifact,
    scene_to_bev,
)

ensure_artifact_dir()
print("project root:", PROJECT_ROOT)
print("artifact directory:", ARTIFACT_DIR)

import numpy as np
import matplotlib.pyplot as plt

state = load_numpy_artifact("03_temporal_state.npz")
rng = np.random.default_rng(9)
current = state["estimate_xy"][-1]
actions = np.array([[0.0, 0.0], [0.5, -0.3], [1.0, -0.8], [1.5, -1.2]])
predicted_next = current[None, :] + actions + rng.normal(0, 0.08, actions.shape)
safety_margin = np.array([4.0, 3.2, 1.7, 0.6])
for action, next_state, margin in zip(actions, predicted_next, safety_margin):
    accepted = margin > 1.0
    print({"action": action.round(2).tolist(), "predicted_next": next_state.round(2).tolist(),
           "safety_margin": float(margin), "accepted": bool(accepted)})

In [ ]:
plt.scatter(predicted_next[:, 0], predicted_next[:, 1], c=safety_margin, cmap="viridis", s=80)
plt.scatter(current[0], current[1], color="red", label="current state")
plt.colorbar(label="safety margin")
plt.legend(); plt.axis("equal"); plt.title("World-action proposals need a safety interface"); plt.show()

练习：加入 action chunk horizon、model uncertainty 和 latency budget；比较 behavior-cloning MSE 与 closed-loop safety；解释为什么 action proposals 不能绕过 Chapter 09 的 independent safety layer。